# Querynode

## Objetivo

Traducir programas escritos en el lenguaje QueryNode a código JavaScript, permitiendo definir endpoints y operaciones SQL mediante una sintaxis simplificada.

## Alcance

* Declarar la configuración de conexión a una base de datos PostgreSQL.
* Definir endpoints REST utilizando los métodos GET, POST, PUT y DELETE.
* Realizar consultas SELECT para obtener registros de una tabla.
* Realizar operaciones INSERT para insertar registros con sus respectivos valores.
* Realizar operaciones UPDATE para modificar registros mediante asignaciones de valores.
* Realizar operaciones DELETE para eliminar registros con diferentes condiciones.
* Seleccionar todas las columnas mediante * o especificar  que columnas seleccionar.
* Utilizar condiciones WHERE con los operadores =, > y <.
* Combinar condiciones mediante los operadores lógicos AND y OR.
* Utilizar parámetros dinámicos declarados en las rutas de los endpoints, como :id.

## Especificaciones léxicas


| **Lexema** | **Token** | **Patrón** |
|---|---|---
| database | DATABASE | `database` |
| host | HOST | `host` |
| port | PORT | `port` |
| user | USER | `user` |
| dbname | DBNAME | `dbname` |
| password | PASSWORD | `password` |
| GET | GET | `GET` |
| POST | POST | `POST` |
| PUT | PUT | `PUT` |
| DELETE | DELETE | `DELETE` |
| SELECT | SELECT | `SELECT` |
| FROM | FROM | `FROM` |
| WHERE | WHERE | `WHERE` |
| INSERT | INSERT | `INSERT` |
| INTO | INTO | `INTO` |
| VALUES | VALUES | `VALUES` |
| UPDATE | UPDATE | `UPDATE` |
| SET | SET | `SET` |
| AND | AND | `AND` |
| OR | OR | `OR` |
| "/api/alumnos/:id" | RUTA | `"/[^"\\\n]*"` |
| "Juan" | STRING | `"([^"\\\n]\|\\.)*"` |
| alumnos | ID | `[a-zA-Z_][a-zA-Z_0-9]*` |
| :id | PARAMETRO | `:[a-zA-Z_][a-zA-Z_0-9]*` |
| 25 | NUMBER | `[0-9]+` |
| , | COMMA | , |
| \* | ASTERISCO | `*` |
| = | IGUAL | `=` |
| > | MAYOR | `>` |
| < | MENOR | `<` |
| { | LKEY |  { |
| } | RKEY | } |
| ( | LPAR | ( |
| ) | RPAR | ) | 

## Especificaciones sintácticas

```bnf
<Programa> ::= <Conexion> <Endpoints>

<Conexion> ::= DATABASE LKEY HOST STRING PORT NUMBER USER STRING DBNAME STRING PASSWORD STRING RKEY

<Endpoints> ::= <Endpoint> | <Endpoint> <Endpoints>

<Endpoint> ::= GET RUTA LKEY <Cuerpo_get> RKEY
             | POST RUTA LKEY <Cuerpo_post> RKEY
             | PUT RUTA LKEY <Cuerpo_put> RKEY
             | DELETE RUTA LKEY <Cuerpo_delete> RKEY

<Cuerpo_get> ::= SELECT <Select_list> FROM ID <Where_opcional>

<Select_list> ::= ASTERISCO | <Lista_ids>

<Where_opcional> ::= λ | WHERE <Condiciones>

<Cuerpo_post> ::= INSERT INTO ID LPAR <Lista_ids> RPAR VALUES LPAR <Lista_valores> RPAR

<Lista_valores> ::= <Valor_fijo> | <Valor_fijo> COMMA <Lista_valores>

<Cuerpo_put> ::= UPDATE ID SET <Lista_asignaciones> WHERE <Condiciones>

<Lista_asignaciones> ::= <Asignacion> | <Asignacion> COMMA <Lista_asignaciones>

<Asignacion> ::= ID IGUAL <Valor_fijo>

<Cuerpo_delete> ::= DELETE FROM ID WHERE <Condiciones>

<Condiciones> ::= <Conjuncion> | <Conjuncion> OR <Condiciones>

<Conjuncion> ::= <Condicion> | <Condicion> AND <Conjuncion>

<Condicion> ::= ID <Operador> <Valor>

<Operador> ::= IGUAL | MAYOR | MENOR

<Valor> ::= NUMBER | STRING | PARAMETRO

<Valor_fijo> ::= NUMBER | STRING

<Lista_ids> ::= ID | ID COMMA <Lista_ids>
```


## Especificaciones semánticas

| **Regla** | **Acciones semánticas** |
|---|---|
| `<Programa> ::= <Conexion> <Endpoints>` | Obtiene la configuración de conexión y la lista de endpoints. |
| `<Endpoints> ::= <Endpoint>` | Crear una lista con un endpoint. |
| `<Endpoints> ::= <Endpoint> <Endpoints>` | Agregar recursivamente múltiples endpoints a la lista. |
| `<Endpoint> ::= GET RUTA LKEY <Cuerpo_get> RKEY` | Generar una ruta `app.get(...)` para consultar registros. |
| `<Endpoint> ::= POST RUTA LKEY <Cuerpo_post> RKEY` | Generar una ruta `app.post(...)` para insertar registros. |
| `<Endpoint> ::= PUT RUTA LKEY <Cuerpo_put> RKEY` | Generar una ruta `app.put(...)` para actualizar registros. |
| `<Endpoint> ::= DELETE RUTA LKEY <Cuerpo_delete> RKEY` | Generar una ruta `app.delete(...)` para eliminar registros. |
| `<Cuerpo_get> ::= SELECT <Select_list> FROM ID <Where_opcional>` | Construir una consulta SELECT con las columnas, tabla y condiciones opcionales. |
| `<Select_list> ::= ASTERISCO` | Indicar que se seleccionan todas las columnas (*). |
| `<Select_list> ::= <Lista_ids>` | Obtener la lista de columnas seleccionadas. |
| `<Where_opcional> ::= λ` | Indicar que la consulta no tiene condiciones WHERE. |
| `<Where_opcional> ::= WHERE <Condiciones>` | Agregar las condiciones WHERE a la consulta. |
| `<Cuerpo_post> ::= INSERT INTO ID LPAR <Lista_ids> RPAR VALUES LPAR <Lista_valores> RPAR` | Construir INSERT INTO con la tabla, columnas y valores. |
| `<Lista_valores> ::= <Valor>` | Crear una lista con un valor. |
| `<Lista_valores> ::= <Valor> COMMA <Lista_valores>` | Agregar recursivamente valores a la lista. |
| `<Cuerpo_put> ::= UPDATE ID SET <Lista_asignaciones> WHERE <Condiciones>` | Construir UPDATE con la tabla, asignaciones y condiciones WHERE. |
| `<Lista_asignaciones> ::= <Asignacion>` | Crear una lista con una asignación. |
| `<Lista_asignaciones> ::= <Asignacion> COMMA <Lista_asignaciones>` | Agregar recursivamente asignaciones a la lista. |
| `<Asignacion> ::= ID IGUAL <Valor>` | Asociar una columna con el valor que se le asignará. |
| `<Cuerpo_delete> ::= DELETE FROM ID WHERE <Condiciones>` | Construir DELETE FROM con la tabla y las condiciones WHERE obligatorias. |
| `<Condiciones> ::= <Condicion>` | Obtener una condición individual. |
| `<Condiciones> ::= <Condicion> AND <Condiciones>` | Combinar recursivamente condiciones mediante AND. |
| `<Condiciones> ::= <Condicion> OR <Condiciones>` | Combinar recursivamente condiciones mediante OR. |
| `<Condicion> ::= ID <Operador> <Valor>` | Construir una comparación entre una columna y un valor. |
| `<Operador> ::= IGUAL` | Devolver el operador de igualdad (=). |
| `<Operador> ::= MAYOR` | Devolver el operador mayor que (>). |
| `<Operador> ::= MENOR` | Devolver el operador menor que (<). |
| `<Valor> ::= NUMBER` | Convertir y guardar un valor numérico. |
| `<Valor> ::= PARAMETRO` | Identificar un parámetro y registrar su referencia. |
| `<Valor> ::= STRING` | Extraer y guardar un valor de texto. |
| `<Lista_ids> ::= ID` | Crear una lista con el nombre de una columna. |
| `<Lista_ids> ::= ID COMMA <Lista_ids>` | Agregar recursivamente nombres de columnas a la lista. |

## Ejemplos

##### **Programa Fuente:**
```text
database {
    host "localhost"
    port 5432
    user "user"
    dbname "dbname"
    password "password"
}

GET "/api/alumnos/:id" {
    SELECT * FROM alumnos
    WHERE id = :id
}

```

##### **Programa Destino:**

```text
const pool = new Pool({
  host: 'localhost',
  port: 5432,
  user: 'user',
  database: 'dbname',
  password: 'password'
});

app.get('/api/alumnos', async (req, res) => {
 try {
    const result = await pool.query('SELECT * FROM alumnos');
    res.json(result.rows);
  } catch (error) {
    res.status(500).json({ error: error.message });
  }
 });
```

In [27]:
%%file lexer.py

import ply.lex as lex

reservadas = {
    'database': 'DATABASE',
    'host': 'HOST',
    'port': 'PORT',
    'user': 'USER',
    'dbname': 'DBNAME',
    'password': 'PASSWORD',
    'get': 'GET',
    'post': 'POST',
    'put': 'PUT',
    'delete': 'DELETE',
    'select': 'SELECT',
    'from': 'FROM',
    'where': 'WHERE',
    'insert': 'INSERT',
    'into': 'INTO',
    'values': 'VALUES',
    'update': 'UPDATE',
    'set': 'SET',
    'and': 'AND',
    'or': 'OR'
}

tokens = [
    'ID',
    'STRING',
    'RUTA',
    'PARAMETRO',
    'NUMBER',
    'LKEY',
    'RKEY',
    'LPAR',
    'RPAR',
    'COMMA',
    'ASTERISCO',
    'IGUAL',
    'MAYOR',
    'MENOR'
] + list(reservadas.values())



t_LKEY = r'\{'
t_RKEY = r'\}'
t_LPAR = r'\('
t_RPAR = r'\)'
t_COMMA = r','
t_ASTERISCO = r'\*'
t_IGUAL = r'='
t_MAYOR = r'>'
t_MENOR = r'<'



def t_RUTA(t):
    r'"/[^"\n]*"'
    t.value = t.value[1:-1]
    return t


def t_STRING(t):
    r'"([^"\\\n]|\\.)*"'
    t.value = t.value[1:-1]
    return t


def t_PARAMETRO(t):
    r':[a-zA-Z_][a-zA-Z_0-9]*'
    return t


def t_NUMBER(t):
    r'[0-9]+'
    t.value = int(t.value)
    return t



def t_ID(t):
    r'[a-zA-Z_][a-zA-Z_0-9]*'
    t.type = reservadas.get(t.value.lower(), 'ID')
    return t

t_ignore = ' \t\r'


def t_newline(t):
    r'\n+'
    t.lexer.lineno += len(t.value)


def t_error(t):
    raise ValueError(
        f"Error léxico en línea {t.lineno}: {t.value[0]}"
    )


lexer = lex.lex()

Overwriting lexer.py


In [28]:
%%file generar_endpoint.py

import ollama

def generar_endpoint(conexion, endpoints):

    datos = ""

    for metodo, ruta, cuerpo in endpoints:
        datos += f"""
Método: {metodo}
Ruta: {ruta}
SQL: {cuerpo}
"""

    respuesta = ollama.chat(
        model='qwen2.5-coder:7b',
        messages=[
            {
                'role': 'system',
                'content': '''
Eres un traductor literal de SQL a JavaScript con Express.

Devuelve solo el Javascript sin markdown.
Respeta exactamente los métodos, rutas y consultas SQL.
No agregues operaciones ni cláusulas SQL.

Para POST, ejecuta el INSERT original directamente con pool.query().
No utilices req.body.

Para GET, PUT y DELETE, conserva los valores fijos.
Solo convierte los parámetros dinámicos como :id a $1 usando req.params.
'''
            },
            {
                'role': 'user',
                'content': f'''
Genera un servidor Express con una conexión pg.Pool
y app.listen(3000).
Genera únicamente los endpoints proporcionados.
Utiliza pool.query() y maneja los errores.

Conexión: {conexion}

Endpoints:
{datos}
'''
            }
        ],
        options={'temperature': 0}
    )

    codigo = respuesta['message']['content'].strip()

    if codigo.startswith("```"):
        codigo = codigo.split("\n", 1)[1]

    if codigo.endswith("```"):
        codigo = codigo.rsplit("```", 1)[0]

    return codigo.strip()

Overwriting generar_endpoint.py


In [29]:
%%file parser.py

import re
import ply.yacc as yacc

from lexer import tokens, lexer
from generar_endpoint import generar_endpoint



def convertir_valor(tipo, valor):
    if tipo == 'STRING':
        return "'" + valor.replace("'", "''") + "'"

    return str(valor)



def p_programa(p):
    'programa : conexion endpoints'

    conexion = p[1]
    endpoints = p[2]

    p[0] = generar_endpoint(conexion, endpoints)



def p_conexion(p):
    'conexion : DATABASE LKEY HOST STRING PORT NUMBER USER STRING DBNAME STRING PASSWORD STRING RKEY'

    p[0] = {
        'host': p[4],
        'port': p[6],
        'user': p[8],
        'database': p[10],
        'password': p[12]
    }



def p_endpoints(p):
    '''
    endpoints : endpoint
              | endpoint endpoints
    '''

    if len(p) == 2:
        p[0] = [p[1]]
    else:
        p[0] = [p[1]] + p[2]


def p_endpoint(p):
    '''
    endpoint : GET RUTA LKEY cuerpo_get RKEY
             | POST RUTA LKEY cuerpo_post RKEY
             | PUT RUTA LKEY cuerpo_put RKEY
             | DELETE RUTA LKEY cuerpo_delete RKEY
    '''

    metodo = p[1].upper()
    ruta = p[2]
    cuerpo = p[4]

    parametros_ruta = set(
        re.findall(r':([a-zA-Z_][a-zA-Z_0-9]*)', ruta)
    )

    parametros_where = cuerpo['parametros_where']


    if metodo == 'POST':
        if parametros_ruta:
            raise ValueError(
                "Error semantico: POST no admite parametros en la ruta"
            )

    else:
        if parametros_ruta != parametros_where:
            raise ValueError(
                f"Error semantico en {metodo}: "
                f"parámetros de ruta {sorted(parametros_ruta)} "
                f"no coinciden con los del WHERE "
                f"{sorted(parametros_where)}"
            )


    p[0] = (metodo, ruta, cuerpo['sql'])



def p_cuerpo_get(p):
    'cuerpo_get : SELECT select_list FROM ID where_opcional'

    p[0] = {
        'sql': f"SELECT {p[2]} FROM {p[4]}{p[5]['sql']}",
        'parametros_where': p[5]['parametros']
    }


def p_select_list(p):
    '''
    select_list : ASTERISCO
                | lista_ids
    '''

    p[0] = p[1]


def p_where_opcional(p):
    '''
    where_opcional : vacio
                   | WHERE condiciones
    '''

    if len(p) == 2:
        p[0] = {
            'sql': '',
            'parametros': set()
        }
    else:
        p[0] = {
            'sql': f" WHERE {p[2]['sql']}",
            'parametros': p[2]['parametros']
        }


def p_vacio(p):
    'vacio :'
    p[0] = ''



def p_cuerpo_post(p):
    'cuerpo_post : INSERT INTO ID LPAR lista_ids RPAR VALUES LPAR lista_valores RPAR'

    p[0] = {
        'sql': f"INSERT INTO {p[3]} ({p[5]}) VALUES ({p[9]})",
        'parametros_where': set()
    }


def p_lista_valores(p):
    '''
    lista_valores : valor_fijo
                  | valor_fijo COMMA lista_valores
    '''

    if len(p) == 2:
        p[0] = p[1]
    else:
        p[0] = f"{p[1]}, {p[3]}"



def p_cuerpo_put(p):
    'cuerpo_put : UPDATE ID SET lista_asignaciones WHERE condiciones'

    p[0] = {
        'sql': f"UPDATE {p[2]} SET {p[4]} WHERE {p[6]['sql']}",
        'parametros_where': p[6]['parametros']
    }


def p_lista_asignaciones(p):
    '''
    lista_asignaciones : asignacion
                       | asignacion COMMA lista_asignaciones
    '''

    if len(p) == 2:
        p[0] = p[1]
    else:
        p[0] = f"{p[1]}, {p[3]}"


def p_asignacion(p):
    'asignacion : ID IGUAL valor_fijo'

    p[0] = f"{p[1]} = {p[3]}"



def p_cuerpo_delete(p):
    'cuerpo_delete : DELETE FROM ID WHERE condiciones'

    p[0] = {
        'sql': f"DELETE FROM {p[3]} WHERE {p[5]['sql']}",
        'parametros_where': p[5]['parametros']
    }



def p_condiciones(p):
    '''
    condiciones : conjuncion
                | conjuncion OR condiciones
    '''

    if len(p) == 2:
        p[0] = p[1]
    else:
        p[0] = {
            'sql': f"({p[1]['sql']} OR {p[3]['sql']})",
            'parametros': (
                p[1]['parametros'] | p[3]['parametros']
            )
        }



def p_conjuncion(p):
    '''
    conjuncion : condicion
               | condicion AND conjuncion
    '''

    if len(p) == 2:
        p[0] = p[1]
    else:
        p[0] = {
            'sql': f"({p[1]['sql']} AND {p[3]['sql']})",
            'parametros': (
                p[1]['parametros'] | p[3]['parametros']
            )
        }



def p_condicion(p):
    'condicion : ID operador valor'

    p[0] = {
        'sql': f"{p[1]} {p[2]} {p[3]['sql']}",
        'parametros': p[3]['parametros']
    }



def p_operador(p):
    '''
    operador : IGUAL
             | MAYOR
             | MENOR
    '''

    p[0] = p[1]



def p_valor(p):
    '''
    valor : NUMBER
          | STRING
          | PARAMETRO
    '''

    tipo = p.slice[1].type

    p[0] = {
        'sql': convertir_valor(tipo, p[1]),
        'parametros': (
            {p[1][1:]} if tipo == 'PARAMETRO' else set()
        )
    }


def p_valor_fijo(p):
    '''
    valor_fijo : NUMBER
               | STRING
    '''

    p[0] = convertir_valor(p.slice[1].type, p[1])



def p_lista_ids(p):
    '''
    lista_ids : ID
              | ID COMMA lista_ids
    '''

    if len(p) == 2:
        p[0] = p[1]
    else:
        p[0] = f"{p[1]}, {p[3]}"



def p_error(p):
    if p:
        raise SyntaxError(
            f"Error sintactico en línea {p.lineno}: {p.value}"
        )
    else:
        raise SyntaxError("Error sintactico: fin inesperado")



parser = yacc.yacc(start='programa')


def analizar(codigo):
    lexer.lineno = 1
    return parser.parse(codigo, lexer=lexer)


Overwriting parser.py


In [30]:
%%file main.py

import sys
from parser import analizar

archivo = sys.argv[1]

entrada = open(archivo, encoding="utf-8").read()

codigo_js = analizar(entrada)

open(archivo.replace(".qnode", ".js"), "w", encoding="utf-8").write(codigo_js)

print("Archivo JavaScript generado")

Overwriting main.py


#### **Ejemplo Simple**

In [31]:
%%file ejemplo_simple.qnode

database {
    host "localhost"
    port 5432
    user "postgres"
    dbname "querynode_db"
    password "1234"
}

GET "/api/alumnos" {
    SELECT * FROM alumnos
}


Overwriting ejemplo_simple.qnode


In [32]:
!py main.py ejemplo_simple.qnode

Archivo JavaScript generado


Generating LALR tables


#### **Ejemplo Complejo**

In [33]:
%%file ejemplo_complejo.qnode

database {
    host "localhost"
    port 5432
    user "postgres"
    dbname "querynode_db"
    password "1234"
}

GET "/api/alumnos" {
    SELECT * FROM alumnos
    WHERE edad < 25 AND activo = 1
}


GET "/api/alumnos/:id" {
    SELECT nombre, apellido, edad, curso, activo FROM alumnos
    WHERE id = :id
}

POST "/api/alumnos" {
    INSERT INTO alumnos (nombre, apellido, edad, curso, activo)
    VALUES ("Juan", "Perez", 20, 5, 1)
}

PUT "/api/alumnos/:id" {
    UPDATE alumnos
    SET edad = 26, curso = 5, activo = 0
    WHERE id = :id 
}

DELETE "/api/alumnos/:id" {
    DELETE FROM alumnos
    WHERE id = :id
}


Overwriting ejemplo_complejo.qnode


In [34]:
!py main.py ejemplo_complejo.qnode

Archivo JavaScript generado


#### **Ejemplo Erroneo**

In [36]:
%%file ejemplo_error.qnode

database {
    host "localhost"
    port 5432
    user "postgres"
    dbname "querynode_db"
    password "1234"
}


GET "/api/alumnos/:id" {
    SELECT * FROM alumnos WHERE id = :codigo
}

Overwriting ejemplo_error.qnode


In [37]:
!py main.py ejemplo_error.qnode

Traceback (most recent call last):
  File "C:\Users\Matias\tp_traductor\main.py", line 9, in <module>
    codigo_js = analizar(entrada)
                ^^^^^^^^^^^^^^^^^
  File "C:\Users\Matias\tp_traductor\parser.py", line 306, in analizar
    return parser.parse(codigo, lexer=lexer)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Matias\AppData\Roaming\Python\Python312\site-packages\ply\yacc.py", line 333, in parse
    return self.parseopt_notrack(input, lexer, debug, tracking, tokenfunc)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Matias\AppData\Roaming\Python\Python312\site-packages\ply\yacc.py", line 1120, in parseopt_notrack
    p.callable(pslice)
  File "C:\Users\Matias\tp_traductor\parser.py", line 80, in p_endpoint
    raise ValueError(
ValueError: Error semantico en GET: parámetros de ruta ['id'] no coinciden con los del WHERE ['codigo']


In [46]:

%%file tests.py

import subprocess
import sys
from pathlib import Path

CARPETA = Path(__file__).resolve().parent

CONEXION = '''
database {
    host "localhost"
    port 5432
    user "postgres"
    dbname "querynode_db"
    password "1234"
}
'''


def ejecutar(tmp_path, codigo):
    archivo = tmp_path / "caso.qnode"
    archivo.write_text(CONEXION + codigo, encoding="utf-8")

    resultado = subprocess.run(
        [sys.executable, str(CARPETA / "main.py"), str(archivo)],
        capture_output=True,
        text=True,
        cwd=CARPETA
    )

    generado = tmp_path / "caso.js"

    return resultado, generado


def test_01_get_simple(tmp_path):
    resultado, generado = ejecutar(tmp_path, '''
GET "/api/alumnos" {
    SELECT * FROM alumnos
}
''')

    assert resultado.returncode == 0, resultado.stderr
    assert generado.exists()

    actual = generado.read_text(encoding="utf-8")
    assert "app.get(" in actual
    assert "SELECT * FROM alumnos" in actual


def test_02_get_parametro(tmp_path):
    resultado, generado = ejecutar(tmp_path, '''
GET "/api/alumnos/:id" {
    SELECT nombre, edad FROM alumnos
    WHERE id = :id
}
''')

    assert resultado.returncode == 0, resultado.stderr
    assert generado.exists()

    actual = generado.read_text(encoding="utf-8")
    assert "/api/alumnos/:id" in actual
    assert "$1" in actual
    assert "req.params" in actual


def test_03_post(tmp_path):
    resultado, generado = ejecutar(tmp_path, '''
POST "/api/alumnos" {
    INSERT INTO alumnos (nombre, edad)
    VALUES ("Camila", 23)
}
''')

    assert resultado.returncode == 0, resultado.stderr
    assert generado.exists()

    actual = generado.read_text(encoding="utf-8")
    assert "app.post(" in actual
    assert "INSERT INTO alumnos" in actual
    assert "Camila" in actual
    assert "23" in actual


def test_04_put(tmp_path):
    resultado, generado = ejecutar(tmp_path, '''
PUT "/api/alumnos/:id" {
    UPDATE alumnos
    SET edad = 26, curso = 4
    WHERE id = :id
}
''')

    assert resultado.returncode == 0, resultado.stderr
    assert generado.exists()

    actual = generado.read_text(encoding="utf-8")
    assert "app.put(" in actual
    assert "UPDATE alumnos" in actual
    assert "edad = 26" in actual
    assert "curso = 4" in actual


def test_05_delete(tmp_path):
    resultado, generado = ejecutar(tmp_path, '''
DELETE "/api/alumnos/:id" {
    DELETE FROM alumnos
    WHERE id = :id AND activo = 0
}
''')

    assert resultado.returncode == 0, resultado.stderr
    assert generado.exists()

    actual = generado.read_text(encoding="utf-8")
    assert "app.delete(" in actual
    assert "DELETE FROM alumnos" in actual
    assert "activo = 0" in actual


def test_06_varios_endpoints(tmp_path):
    resultado, generado = ejecutar(tmp_path, '''
GET "/api/alumnos" {
    SELECT * FROM alumnos
}

POST "/api/alumnos" {
    INSERT INTO alumnos (nombre)
    VALUES ("Ana")
}
''')

    assert resultado.returncode == 0, resultado.stderr
    assert generado.exists()

    actual = generado.read_text(encoding="utf-8")

    assert "app.get(" in actual
    assert "app.post(" in actual
    assert "SELECT * FROM alumnos" in actual
    assert "INSERT INTO alumnos" in actual
    assert "Ana" in actual



def test_07_error_lexico(tmp_path):
    resultado, generado = ejecutar(tmp_path, '''
GET "/api/alumnos" {
    SELECT * FROM alumnos @
}
''')

    assert resultado.returncode != 0
    assert not generado.exists()
    assert "Error léxico" in resultado.stderr


def test_08_error_sintactico(tmp_path):
    resultado, generado = ejecutar(tmp_path, '''
GET "/api/alumnos" {
    SELECT nombre alumnos
}
''')

    assert resultado.returncode != 0
    assert not generado.exists()
    assert "Error sintactico" in resultado.stderr


def test_09_error_semantico(tmp_path):
    resultado, generado = ejecutar(tmp_path, '''
GET "/api/alumnos/:id" {
    SELECT * FROM alumnos WHERE id = :codigo
}
''')

    assert resultado.returncode != 0
    assert not generado.exists()
    assert "Error semantico" in resultado.stderr


def test_10_error_post(tmp_path):
    resultado, generado = ejecutar(tmp_path, '''
POST "/api/alumnos/:id" {
    INSERT INTO alumnos (nombre, edad)
    VALUES ("Ana", 20)
}
''')

    assert resultado.returncode != 0
    assert  generado.exists()
    assert "Error semantico" in resultado.stderr


Overwriting tests.py


In [47]:
!pytest tests.py

============================= test session starts =============================
platform win32 -- Python 3.14.6, pytest-9.0.3, pluggy-1.6.0
rootdir: C:\Users\Matias\tp_traductor
plugins: anyio-4.12.1
collected 10 items

tests.py .........F                                                      [100%]

================================== FAILURES ===================================
_____________________________ test_10_error_post ______________________________

tmp_path = WindowsPath('C:/Users/Matias/AppData/Local/Temp/pytest-of-Matias/pytest-23/test_10_error_post0')

    def test_10_error_post(tmp_path):
        resultado, generado = ejecutar(tmp_path, '''
    POST "/api/alumnos/:id" {
        INSERT INTO alumnos (nombre, edad)
        VALUES ("Ana", 20)
    }
    ''')
    
        assert resultado.returncode != 0
>       assert  generado.exists()
E       AssertionError: assert False
E        +  where False = exists()
E        +    where exists = WindowsPath('C:/Users/Matias/AppData/Local

## Conclusiones

Con este trabajo pude desarrollar QueryNode, un traductor que permite crear endpoints REST y consultas SQL de forma más sencilla, generando código JavaScript.

Utilice Python con PLY para realizar los análisis léxico, sintáctico y semántico, y Ollama para generar el código, tambien hice 10 pruebas para comprobar su funcionamiento.

Durante el desarrollo fui entendiendo mejor cómo funciona cada etapa de un traductor y cómo se relacionan entre sí. Algunas partes me costaron más que otras, sobre todo al momento de definir la gramática y hacer que el parser reconociera correctamente las instrucciones, al igual que usar el agente inteligente ya que no generaba lo que yo necesitaba, pero pude solucionarlo.

En general, me sirvió para poner en práctica lo que vimos en la cursada y aprender a utilizar herramientas que antes no conocía tanto. Todavia hay cosas que se podrían mejorar, como agregar mas consultas SQL y nuevas validaciones, pero estoy conforme con lo que pude desarrollar.